In [24]:
from spark_utils import SparkUtils

spark_url = "spark://spark-master:7077"
app_name = "SparkSQL Example 1"
su = SparkUtils(spark_url, app_name)
su

<SparkContext master=spark://spark-master:7077 appName=SparkSQL Example 1>

In [25]:
from pyspark.sql.types import StructField, StringType, StructType, IntegerType

data = [
(1, "Alice", 29),
(2, "Bob", 35),
(3, "Charlie", 41)
]

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("age", IntegerType())
])

df = su._spark.createDataFrame(data, schema)
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)



In [26]:
from pyspark.sql.types import StructField, StringType, FloatType, TimestampType
from datetime import datetime

factory_data = [
("M001", datetime(2026, 1, 26, 8, 0, 0), 75.3),
("M002", datetime(2026, 1, 26, 8, 5, 0), 68.7),
("M001", datetime(2026, 1, 26, 8, 10, 0), 76.1),
("M003", datetime(2026, 1, 26, 8, 15, 0), 72.4),
("M002", datetime(2026, 1, 26, 8, 20, 0), 69.8),
("M001", datetime(2026, 1, 26, 8, 25, 0), 77.5),
("M003", datetime(2026, 1, 26, 8, 30, 0), 73.2),
("M002", datetime(2026, 1, 26, 8, 35, 0), 70.1),
("M001", datetime(2026, 1, 26, 8, 40, 0), 78.0),
("M003", datetime(2026, 1, 26, 8, 45, 0), 74.6),
]

factory_schema = StructType([
    StructField("id", StringType()),
    StructField("date", TimestampType()),
    StructField("temp", FloatType())
])

#1. Explore the schema of the DataFrame
df_factory = su._spark.createDataFrame(factory_data, factory_schema)
df_factory.printSchema()

root
 |-- id: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- temp: float (nullable = true)



In [27]:
print(f"Factory Data: {df_factory.count()}")

Factory Data: 10


In [28]:
from pyspark.sql.functions import col
#4. Filter records above a temperature threshold (temp > 75)
df_filtered = df_factory.filter(col("temp") > 75)
df_filtered.show(truncate=False)

+----+-------------------+----+
|id  |date               |temp|
+----+-------------------+----+
|M001|2026-01-26 08:00:00|75.3|
|M001|2026-01-26 08:10:00|76.1|
|M001|2026-01-26 08:25:00|77.5|
|M001|2026-01-26 08:40:00|78.0|
+----+-------------------+----+



In [29]:
from pyspark.sql.functions import col
df_filtered = df_filtered.orderBy(col("temp"), ascending=False).limit(3)
df_filtered.show()

+----+-------------------+----+
|  id|               date|temp|
+----+-------------------+----+
|M001|2026-01-26 08:40:00|78.0|
|M001|2026-01-26 08:25:00|77.5|
|M001|2026-01-26 08:10:00|76.1|
+----+-------------------+----+



In [30]:
from pyspark.sql.functions import col
#5. Count the number of readings per machine
df_count = df_factory.groupBy(col("id")).count()
df_count.show()

+----+-----+
|  id|count|
+----+-----+
|M002|    3|
|M001|    4|
|M003|    3|
+----+-----+



In [31]:
from pyspark.sql.functions import col, avg, min, max
#2. Get the average temperature per machine
#3. Find the maximum and minimum temperature per machine
df_avg = df_factory.groupBy(col("id")).agg(
    avg("temp").alias("avg_temp"),
    min("temp").alias("min_temp"),
    max("temp").alias("max_temp")
)
df_avg.show()

+----+-----------------+--------+--------+
|  id|         avg_temp|min_temp|max_temp|
+----+-----------------+--------+--------+
|M002|69.53333282470703|    68.7|    70.1|
|M003|73.39999898274739|    72.4|    74.6|
|M001|76.72500038146973|    75.3|    78.0|
+----+-----------------+--------+--------+



In [23]:
#6. Find the machine with the highest temperature
from pyspark.sql.functions import col
df_max_temp = df_factory.orderBy(col("temp").desc()).limit(1)
df_max_temp.show()

+----+-------------------+----+
|  id|               date|temp|
+----+-------------------+----+
|M001|2026-01-26 08:40:00|78.0|
+----+-------------------+----+

